In [72]:
# Load Fiji
import geopandas as gpd
from odc.geo.geom import Geometry

fiji_tiled_geoms = gpd.read_file(
    "/Users/wj/Projects/csdr/csdr-cloud-spatial/debug_geoms_tiled.geojson"
)

print(fiji_tiled_geoms["geometry"][0].bounds)

fiji_tiled_geoms = [Geometry(geom) for geom in fiji_tiled_geoms["geometry"]]

geometry_4326 = fiji_tiled_geoms[0].assign_crs("EPSG:4326")
geom_geojson = geometry_4326.geojson(simplify=0)["geometry"]

(-180.0, -24.3819587198891, -176.267461612063, -12.820507703401365)


In [73]:
# Search and load STAC
from pystac import ItemCollection
from rustac import search_sync as rustac_search_sync

dataset_url = "../cache/datasets/seagrass/0-0-1/dep_s2_seagrass.parquet"

items = rustac_search_sync(
    dataset_url,
    intersects=geom_geojson,
    datetime="2017",
)

print(f"Found {len(items)} items")

items = ItemCollection(items)

ok_items = []
naughty_items = []
for item in items:
    print(item.bbox)
    xmin, ymin, xmax, ymax = item.bbox
    if xmin == -180.0 and xmax == 180.0:
        naughty_items.append(item)
    else:
        ok_items.append(item)

print(f"Found {len(ok_items)} ok items and {len(naughty_items)} naughty items")

print("Ok items:")
for item in ok_items:
    print(item.id, item.bbox)

print("Naughty items:")
for item in naughty_items:
    print(item.id, item.bbox)

Found 12 items
[-178.30743677626327, -20.11524267928007, -177.4450541035085, -19.298529078932667]
[-179.169819449018, -16.824114445989782, -178.30743677626327, -15.991730594839629]
[-179.169819449018, -17.65281330691017, -178.30743677626327, -16.824114445989782]
[-179.169819449018, -18.477669425954502, -178.30743677626327, -17.65281330691017]
[-179.169819449018, -19.298529078932667, -178.30743677626327, -18.477669425954502]
[-179.169819449018, -20.927664878354115, -178.30743677626327, -20.11524267928007]
[-179.169819449018, -21.73565465582795, -178.30743677626327, -20.927664878354115]
[-180.0, -15.991730594839629, 180.0, -15.15582341258985]
[-180.0, -16.824114445989782, 180.0, -15.991730594839629]
[-180.0, -17.65281330691017, 180.0, -16.824114445989782]
[-180.0, -18.477669425954502, 180.0, -17.65281330691017]
[-180.0, -19.298529078932667, 180.0, -18.477669425954502]
Found 7 ok items and 5 naughty items
Ok items:
dep_s2_seagrass_068_018_2017 [-178.30743677626327, -20.11524267928007, -17

In [74]:
# What are the naughty items bounds in 3832?
from pyproj import Transformer

t = Transformer.from_crs(4326, 3832, always_xy=True)
antimeridian_3832, _ = t.transform(180, 0)
print(antimeridian_3832)

3339584.723798206


In [75]:
geom_3832 = geometry_4326.to_crs("EPSG:3832")

geom_3832_bbox = geom_3832.boundingbox
print(geom_3832_bbox.left)
print(antimeridian_3832)
print(geom_3832_bbox.right)

print("Geom's left is on the antimeridian and its right is in the western hemisphere.")

# Not a good test due to floating point imprecision.
# if geom_3832_bbox.left < antimeridian_3832 and geom_3832_bbox.right < antimeridian_3832:
#     print("The geometry is on the left side of the antimeridian in 3832")
# elif geom_3832_bbox.left > antimeridian_3832 and geom_3832_bbox.right > antimeridian_3832:
#     print("The geometry is on the right side of the antimeridian in 3832")
# else:
#     print("The geometry crosses the antimeridian in 3832")

3339584.7237982033
3339584.723798206
3755088.9965096987
Geom's left is on the antimeridian and its right is in the western hemisphere.


In [76]:
# The whole of column 66 in the COG grid is straddling the antimeridian.
# Devilish behaviour!
from odc.stac import load
from pyproj import Transformer

print(
    "This geom is just in the western hemisphere. These COGs in 6933 are in both hemispheres, so they have a huge dataset (x dimension) spanning the globe."
)

# Now to try and load these to get a count in 6933.
data = load(naughty_items, crs="EPSG:6933", resolution=10, chunks={})
print(f"Loaded data with shape {data.dims}")

This geom is just in the western hemisphere. These COGs in 6933 are in both hemispheres, so they have a huge dataset (x dimension) spanning the globe.
Loaded data with shape FrozenMappingWarningOnValuesAccess({'y': 50528, 'x': 3473423, 'time': 1})


In [77]:
# Attempted solution:
# Load naughty items in 3832, clip to geometry bbox so we discard
# the eastern-hemisphere portion, mask to geometry, filter indicator values,
# then write a COG for inspection.
from odc.geo.cog import write_cog

indicator = "seagrass"
value_list = [1.0]

# Load in 3832 (Pacific-centered)
data_3832 = load(naughty_items, crs="EPSG:3832", resolution=10, chunks={})
print(f"Loaded data with shape {data_3832.dims}")

# Reproject geometry to 3832
geom_3832 = geometry_4326.to_crs("EPSG:3832")

geom_3832_bbox = geom_3832.boundingbox
print(geom_3832_bbox)

data_3832_clipped = data_3832.rio.clip_box(
    minx=geom_3832_bbox.left,
    miny=geom_3832_bbox.bottom,
    maxx=geom_3832_bbox.right,
    maxy=geom_3832_bbox.top,
    crs="EPSG:3832",
)

# The clip_box left edge is at 3339580.0 (pixel centre) which means the
# geobox extends to 3339575.0 (centre - half pixel). The antimeridian is
# at 3339584.7. Drop leading columns whose pixel left edge (centre - 5m)
# is past the antimeridian.
resolution = 10.0
half_px = resolution / 2
data_3832_clipped = data_3832_clipped.isel(
    x=(data_3832_clipped.x.values - half_px) >= antimeridian_3832
)

print(f"Clipped data shape: {data_3832_clipped.dims}")
print(
    f"Clipped x range: {float(data_3832_clipped.x.min()):.1f} to {float(data_3832_clipped.x.max()):.1f}"
)
print(f"Geobox bbox: {data_3832_clipped.odc.geobox.boundingbox}")
print(f"Antimeridian in 3832: {antimeridian_3832:.1f}")

write_cog(
    data_3832_clipped["seagrass"],
    "naughty_cogs_clipped_3832.tif",
    dtype="float32",
    nodata=0,
    overwrite=True,
)

Loaded data with shape FrozenMappingWarningOnValuesAccess({'y': 48000, 'x': 9600, 'time': 1})
BoundingBox(left=3339584.7237982033, bottom=-2782387.936854609, right=3755088.9965096987, top=-1429757.5285600545, crs=CRS('EPSG:3832'))
Clipped data shape: FrozenMappingWarningOnValuesAccess({'y': 48000, 'x': 9241, 'time': 1})
Clipped x range: 3339595.0 to 3431995.0
Geobox bbox: BoundingBox(left=3339590.0, bottom=-2176000.0, right=3432000.0, top=-1696000.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]'))
Antimeridian in 3832: 3339584.7


PosixPath('naughty_cogs_clipped_3832.tif')

In [78]:
print(data_3832_clipped.odc.geobox.boundingbox)
print(data_3832_clipped.odc.geobox.boundingbox.left)
print(antimeridian_3832)
print(data_3832_clipped.odc.geobox.boundingbox.right)

# print("I think the clipped dataset still crosses the antimeridian (when using geom_3832_bbox.left to clip). Should use the antimeridian_3832 to clip it?")

# With antimeridian as clip:
# BoundingBox(left=3339580.0, bottom=-2176000.0, right=3432000.0, top=-1696000.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]'))
# 3339580.0
# 3339584.723798206
# 3432000.0

# With using geom_3832_bbox.left to clip:
# 3339580.0
# 3339584.723798206
# 3432000.0

# For both methods there is 4m of antimeridian crossing. I need to mask 1 pixel.
# After clipping 1 pixel:
# 3339590.0
# 3339584.723798206
# 3432000.0
print(
    f"After clipping 1 pixel, dataset is east of the antimeridian: {data_3832_clipped.odc.geobox.boundingbox.left > antimeridian_3832}"
)

BoundingBox(left=3339590.0, bottom=-2176000.0, right=3432000.0, top=-1696000.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]'))
3339590.0
3339584.723798206
3432000.0
After clipping 1 pixel, dataset is east of the antimeridian: True


In [79]:
from pyproj import Transformer

t = Transformer.from_crs(4326, 6933, always_xy=True)
antimeridian_6933, _ = t.transform(180, 0)
print(antimeridian_6933)

17367530.445161372


In [82]:
# Add a gutter to the 3832 dataset by the antimeridian so that when reprojecting to 6933 it doesn't cross.
gutter_size_px = 3
data_3832_clipped = data_3832_clipped.isel(x=slice(gutter_size_px, None))
print(f"After removing {gutter_size_px}px gutter: {data_3832_clipped.dims}")
print(
    f"x range: {float(data_3832_clipped.x.min()):.1f} to {float(data_3832_clipped.x.max()):.1f}"
)
print(f"Geobox bbox: {data_3832_clipped.odc.geobox.boundingbox}")

write_cog(
    data_3832_clipped["seagrass"],
    "naughty_cogs_clipped_3832_gutter.tif",
    dtype="float32",
    nodata=0,
    overwrite=True,
)

After removing 3px gutter: FrozenMappingWarningOnValuesAccess({'y': 48000, 'x': 9236, 'time': 1})
x range: 3339645.0 to 3431995.0
Geobox bbox: BoundingBox(left=3339640.0, bottom=-2176000.0, right=3432000.0, top=-1696000.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]'))


PosixPath('naughty_cogs_clipped_3832_gutter.tif')

In [83]:
data_6933_clipped = data_3832_clipped.odc.reproject("EPSG:6933", resolution=10)
print(data_6933_clipped.odc.geobox.boundingbox)
print(data_6933_clipped.odc.geobox.boundingbox.left)
print(antimeridian_6933)
print(data_6933_clipped.odc.geobox.boundingbox.right)
print(data_6933_clipped.dims)
assert data_6933_clipped.sizes["x"] < 1_000_000, "Error: X dimension spanning world."
# Error "GEOSException: IllegalArgumentException: Points of LinearRing do not form a closed linestring"
# After clipping 1 pixel to fix the antimeridian crossing, reprojection to 6933 works fine.

data_4326_clipped = data_3832_clipped.odc.reproject("EPSG:4326", resolution=10)
print(data_4326_clipped.odc.geobox.boundingbox)
print(data_4326_clipped.dims)
# 4326 is a problem, but may not worry us.

BoundingBox(left=-17367500.0, bottom=-2416880.0, right=-17287420.0, top=-1911580.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",150],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","3832"]]'))
-17367500.0
17367530.445161372
-17287420.0
FrozenMappingWarningOnValuesAccess({'time': 1, 'y': 50530, 'x': 8008})
BoundingBox(left=-180.0, bottom=-20.0, right=-170.0, top=-10.0, crs=CRS('PROJCS["WGS 84 / PDC Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",1

In [85]:
# Here are my results from debugging earlier:
# # Expecting
# Original value for Fiji at 100m:
# 296,420,000 m²

# 10m Skipping/Ignoring bad COGs
# 286,171,300 m²


def nice_print_area(area_m2: float) -> str:
    if area_m2 >= 1e6:
        return f"{area_m2 / 1e6:.2f} km²"
    else:
        return f"{area_m2:.2f} m²"


print(
    f"Expecting about {nice_print_area(296_420_000 - 286_171_300)} for the naughty COGs"
)

Expecting about 10.25 km² for the naughty COGs


In [86]:
indicator = "seagrass"
value_list = [1.0]

# xx = data_6933_clipped[indicator].notnull().sum().compute()
# print(xx)

da = data_6933_clipped[indicator].where(data_6933_clipped[indicator].isin(value_list))

count = float(da.notnull().sum().values)
print(f"Counted {count} pixels with value in {value_list}")
one_pixel_area_m2 = abs(
    data_6933_clipped.odc.geobox.resolution.x
    * data_6933_clipped.odc.geobox.resolution.y
)
area_m2 = round(count * one_pixel_area_m2, 2)
print(nice_print_area(area_m2))

Counted 115423.0 pixels with value in [1.0]
11.54 km²


In [ ]:
# from csdr.utils import xarray_calculate_area_m2

# indicator = "seagrass"
# value_list = [1.0]

# total_area_m2 = xarray_calculate_area_m2(
#     data[indicator], geometry_4326, indicator=indicator, value_list=value_list
# )
# print(f"Naughty COGs seagrass area: {nice_print_area(total_area_m2)}")